# Library calling

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from time import sleep
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import datetime


import warnings
# Ignore all warnings (not recommended in general)
warnings.filterwarnings("ignore")

# Defining the Product Information and Location

In [2]:
SummaryFolder=r'C:\Users\Vikram.Vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\99_Automation_Projects'
summaryFile='Scraping_List.txt'
st=pd.read_csv(SummaryFolder+'\\'+summaryFile)
print(st)
#Define Product to extract
search_text = st['Product Name'][9]
print(search_text)
Source="CarParts"
OFolder=fr'C:\Users\Vikram.Vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\99_Automation_Projects\{Source}\Outputs'
IFolder=fr'C:\Users\Vikram.Vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\99_Automation_Projects\{Source}\Inputs'
filename=Source+'ProductLinks_'+search_text+'.xlsx'
df1=pd.read_excel(IFolder+'\\'+filename)
df1

                                  Product Name
0                                Ignition Coil
1                      Windshield Washer Pumps
2                        coupler trailer locks
3          Adjustable Trailer Hitch Ball Mount
4                               Vacuum Cleaner
5                                   Spark Plug
6              bluetooth Enabled Trailer locks
7                          Power Steering Hose
8   Power Steering Pressure Line Hose Assembly
9                       Washer Fluid Reservoir
10          Hitch Ball Mount with Weight Scale
11                               LED Headlamps
12                             LED flashlights
13                   Fiberglass Tonneau Covers
14                    Aluminium Tonneau Covers
15                     Hardfold Tonneau Covers
16                            Spark Plug Wires
17                 washer fluid reservoir tank
18                   Non Automotive Gas Struts
Washer Fluid Reservoir


,Links
0,https://www.carparts.com/washer-reservoir/repl...
1,https://www.carparts.com/washer-reservoir/repl...
2,https://www.carparts.com/washer-reservoir/repl...
3,https://www.carparts.com/washer-reservoir/repl...
4,https://www.carparts.com/washer-reservoir/repl...
...,...
265,https://www.carparts.com/washer-reservoir/repl...
266,https://www.carparts.com/washer-reservoir/repl...
267,https://www.carparts.com/washer-reservoir/repl...
268,https://www.carparts.com/washer-reservoir/repl...


In [3]:
links=df1['Links']
length=len(links)
print(length)
print(links[71])

270
https://www.carparts.com/washer-reservoir/replacement/rept370540


# Setting Webdriver and Website Specific Information

In [4]:
path= 'C://chromedriver.exe'
driver=webdriver.Chrome()
driver.maximize_window()

# Defining the Dataframe and Extracting the data into the Dataframe

In [5]:
cols =['Sl.No','Attributes'] #
df = pd.DataFrame(columns=cols)
df
count=0

In [6]:
from IPython.display import clear_output 
import numpy as np
from datetime import timedelta
import datetime 
timestamp=[]
timediff=[]
def timeremaining(balanceitem,i):
    timestamp.insert(i,datetime.datetime.now())
    if i>=1:
        diff=timestamp[i]-timestamp[i-1]
        timediff.insert(i,diff.total_seconds())		
        remainingtime=np.median(timediff)*balanceitem
        millis=int(remainingtime)		
        #print(millis)
        seconds=(millis)%60
        seconds = int(seconds)
        minutes=(millis/(60))%60
        minutes = int((minutes)) #math.floor
        hours=(millis/(60*60))%24
        hours=int(hours)
        print(f'Item looped: {i+1} \nItems Remining: {balanceitem-1} and \nTime Remaining: {hours}:{minutes}:{seconds}')
        etc=datetime.datetime.now()+timedelta(hours=hours,minutes=minutes,seconds=seconds)
        print(f'Estimated time of completion:{etc}')        
        clear_output(wait=True)

In [7]:
for i in range(length):
    driver.get(links[i])
    df.loc[count,'Sl.No']=i
    sleep(2)
    df.loc[count,"Part Number"]=driver.find_element(By.CSS_SELECTOR,'[class="StyledBox-sc-13pk1d4-0 dbloPH"]').text.split("#")[1]
    product = driver.find_element(By.ID,"skuTitle")
    if len(product.text)==0:
        df.loc[count,'Name']="-"
    else:
        df.loc[count,'Name']=product.text.replace('Replacement\n','')
    price=driver.find_element(By.CLASS_NAME,'sc-pbej5q-0')
    if len(price.text)==0:
        df.loc[count,'Current Price']="-"
    else:    
        df.loc[count,'Current Price']=float(price.text.replace('$',''))
    sleep(1)
    driver.execute_script("window.scrollTo(0, 1000);")
    sleep(1)
    Aelements=driver.find_elements(By.CLASS_NAME,"iOfUvl")
    Tlist=[]
    for element in Aelements:
        Tlist.append(element.text)
    Alist=[]
    for j in range(0, len(Aelements), 2):
        Alist.append(Tlist[j]+Tlist[j+1])
    Alist
    if len(Alist)==0:
        df.loc[count,'Attributes']="-"
    else:
        df.at[count,'Attributes']=Alist
    driver.execute_script("window.scrollTo(0, 2500);")
    reviewscount=driver.find_element(By.CSS_SELECTOR,'[data-testid="desktop-review-count-link-pdp"]').text
    df.loc[count,"No of Ratings"]=int(reviewscount.split(' ')[0].replace('(',''))
    df.loc[count,'Details']=driver.find_elements(By.CLASS_NAME,"jcgGOf")[1].text
    df.loc[count,'Links']=links[i]
    
    try:
        df.loc[count,'Rating']=float(driver.find_element(By.CSS_SELECTOR,'[class="tt-c-reviews-summary__rating-number"]').text)
    except:
        pass
    df.loc[count,'Source']=Source
    count=count+1
    balanceitem=length-i
    timeremaining(balanceitem,i)
#driver.quit()

Item looped: 270 
Items Remining: 0 and 
Time Remaining: 0:0:9
Estimated time of completion:2024-02-09 14:58:21.151908


In [8]:
print(df.shape)
df.head()

(270, 10)


,Sl.No,Attributes,Part Number,Name,Current Price,No of Ratings,Details,Links,Rating,Source
0,0,"[Part:Washer Reservoir, Brand:Replacement, Not...",REPT370533,"Washer Reservoir, With Pump",64.49,1325.0,"Manufactured from top quality components, this...",https://www.carparts.com/washer-reservoir/repl...,4.7,CarParts
1,1,"[Part:Washer Reservoir, Brand:Replacement, Not...",REPC370527,"Washer Reservoir, With Pump",51.49,1325.0,"Manufactured from top quality components, this...",https://www.carparts.com/washer-reservoir/repl...,4.7,CarParts
2,2,"[Part:Washer Reservoir, Brand:Replacement, Not...",REPD370514,"Washer Reservoir, With Pump",43.49,1325.0,"Manufactured from top quality components, this...",https://www.carparts.com/washer-reservoir/repl...,4.7,CarParts
3,3,"[Part:Washer Reservoir, Brand:Replacement, Not...",REPJ370530,"Washer Reservoir, With Pump",66.99,1325.0,"Manufactured from top quality components, this...",https://www.carparts.com/washer-reservoir/repl...,4.7,CarParts
4,4,"[Part:Washer Reservoir, Brand:Replacement, Not...",REPC370572,"Washer Reservoir, With Pump",47.49,1325.0,"Manufactured from top quality components, this...",https://www.carparts.com/washer-reservoir/repl...,4.7,CarParts


# Post Processing Data and Exporting

In [9]:
df['Brand']=df['Attributes'].apply(lambda x: str(x).replace('[','').replace(']','').replace('\'', '').replace(' ','')).str.split('Brand:',expand=True)[1].str.split(',',expand=True)[0]
df['Product']=search_text


In [10]:
cols=["Sl.No",
"Name",
"Product",
"Current Price",
"Rating",
"No of Ratings",
"Details",
"Attributes",
"Brand",
"Part Number",
"Links",
"Source"
]
df = df[cols]
df.head()

,Sl.No,Name,Product,Current Price,Rating,No of Ratings,Details,Attributes,Brand,Part Number,Links,Source
0,0,"Washer Reservoir, With Pump",Washer Fluid Reservoir,64.49,4.7,1325.0,"Manufactured from top quality components, this...","[Part:Washer Reservoir, Brand:Replacement, Not...",Replacement,REPT370533,https://www.carparts.com/washer-reservoir/repl...,CarParts
1,1,"Washer Reservoir, With Pump",Washer Fluid Reservoir,51.49,4.7,1325.0,"Manufactured from top quality components, this...","[Part:Washer Reservoir, Brand:Replacement, Not...",Replacement,REPC370527,https://www.carparts.com/washer-reservoir/repl...,CarParts
2,2,"Washer Reservoir, With Pump",Washer Fluid Reservoir,43.49,4.7,1325.0,"Manufactured from top quality components, this...","[Part:Washer Reservoir, Brand:Replacement, Not...",Replacement,REPD370514,https://www.carparts.com/washer-reservoir/repl...,CarParts
3,3,"Washer Reservoir, With Pump",Washer Fluid Reservoir,66.99,4.7,1325.0,"Manufactured from top quality components, this...","[Part:Washer Reservoir, Brand:Replacement, Not...",Replacement,REPJ370530,https://www.carparts.com/washer-reservoir/repl...,CarParts
4,4,"Washer Reservoir, With Pump",Washer Fluid Reservoir,47.49,4.7,1325.0,"Manufactured from top quality components, this...","[Part:Washer Reservoir, Brand:Replacement, Not...",Replacement,REPC370572,https://www.carparts.com/washer-reservoir/repl...,CarParts


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 270 entries, 0 to 269
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Sl.No          270 non-null    object 
 1   Name           270 non-null    object 
 2   Product        270 non-null    object 
 3   Current Price  270 non-null    float64
 4   Rating         248 non-null    float64
 5   No of Ratings  270 non-null    float64
 6   Details        270 non-null    object 
 7   Attributes     270 non-null    object 
 8   Brand          270 non-null    object 
 9   Part Number    270 non-null    object 
 10  Links          270 non-null    object 
 11  Source         270 non-null    object 
dtypes: float64(3), object(9)
memory usage: 35.5+ KB


In [12]:
dfAtt=df[['Links','Attributes']]
dfAtt=dfAtt.explode('Attributes')
dfAtt[['Attributes', 'Value']] = dfAtt['Attributes'].str.split(':',1, expand=True)

In [13]:
summary_df = pd.pivot_table(dfAtt, values='Value', index='Attributes',aggfunc='count').reset_index()
summary_df =summary_df.sort_values(by='Value',ascending=False)
summary_df=summary_df.rename(columns ={'Value':'No Products contains this Attribute'})
summary_df


,Attributes,No Products contains this Attribute
0,Brand,270
2,Interchange Part Number,270
5,Part,270
6,Prop 65 Warning,270
7,Quantity Sold,270
10,Returns Policy,270
11,Type,270
12,Warranty,270
8,Replaces OE Number,269
9,Replaces Partslink Number,266


In [14]:
with pd.ExcelWriter(OFolder+'\\'+f'{Source}ProductDetails_'+search_text+'.xlsx') as writer:  # doctest: +SKIP
    df.to_excel(writer,index=False, sheet_name='Raw')
    summary_df.to_excel(writer,index=False, sheet_name='Attribute_Summary')

# Archived Codes